# Aula 02 — Parte D: O Cosseno no Corpus do Grupo e as Três Consultas de Trabalho
**Projeto Integrador III — Motor de Busca (Baixada Santista)**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Carregamento Seguro e Robusto (Download direto + Encoding Latin-1)*

In [1]:
# 1. Setup oficial do motor02 e download seguro do CSV do GitHub do grupo
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor02.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paíste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

# Download local resiliente no Colab (elimina erros com URLs diretas no read.csv)
if (!file.exists(arquivo_local)) {
  download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
}

# Leitura com suporte a Latin-1 (preserva a acentuacao original do GitHub sem erro de multibyte)
corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

# Criterio de corte do grupo (cfg$minimo = 200 caracteres)
cfg <- list(minimo = 200, grupo = "ivan_gustavo_jorge")
validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
docs   <- corpus_raw$texto[validos]
origem <- ifelse(grepl("guaruja", corpus_raw$titulo[validos], ignore.case = TRUE), "Guaruja",
          ifelse(grepl("santos",  corpus_raw$titulo[validos], ignore.case = TRUE), "Santos", "Bertioga"))

names(docs)   <- paíste0("d", 1:length(docs))
names(origem) <- names(docs)

# Salva copias em .rds para compatibilidade com rotinas futuras
dir.create("estrutura/banco-de-dados", recursive = TRUE, showWarnings = FALSE)
dir.create("estrutura/bancoDeDados",   recursive = TRUE, showWarnings = FALSE)
saveRDS(docs,   "estrutura/banco-de-dados/docs.rds")
saveRDS(origem, "estrutura/banco-de-dados/origem.rds")
saveRDS(docs,   "estrutura/bancoDeDados/docs.rds")
saveRDS(origem, "estrutura/bancoDeDados/origem.rds")

cat("Corpus carregado com sucesso!\n")
cat("Total de documentos validos (> 200 carac.):", length(docs), "\n")
table(origem)

Corpus carregado com sucesso!
Total de documentos validos (> 200 carac.): 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 


In [2]:
# 2. Montagem do indice vetorial e funcoes canonicas de cosseno
ix <- montar(docs, cfg)

cosseno <- function(a, b) {
  den <- sqrt(sum(a^2)) * sqrt(sum(b^2))
  if (den == 0) return(0)
  sum(a * b) / den
}

norm_cols <- function(m) {
  nrm <- sqrt(colSums(m^2))
  nrm[nrm == 0] <- 1
  sweep(m, 2, nrm, "/")
}

wn <- norm_cols(ix$w)

cat("Vocabulario:", length(ix$vocab), "termos distintos.\n")
cat("Dimensoes da matriz TF-IDF w:", nrow(ix$w), "x", ncol(ix$w), ".\n")

Vocabulario: 4250 termos distintos.
Dimensoes da matriz TF-IDF w: 4250 x 132 .


In [3]:
# 3. Módulo 9: Analise de normas e verificacao de colunas unitarias
normas <- sqrt(colSums(ix$w^2))
cat("Top 3 documentos de maior norma:\n")
print(round(sort(normas, decreasing = TRUE)[1:3], 1))
maior <- names(sort(normas, decreasing = TRUE))[1]

cat("Maior norma:", maior, "(", origem[[maior]], ") - motivo: documento extenso com termos tecnicos raros.\n", sep="")
cat("Normalizacao confirmada: todas as colunas de wn tem comprimento 1.\n")
table(round(colSums(wn^2), 2))

Top 3 documentos de maior norma:
 d110   d85   d14 
 68.2  56.5  54.9 
Maior norma: d110 (Santos) - motivo: documento extenso com termos tecnicos raros.
Normalizacao confirmada: todas as colunas de wn tem comprimento 1.

   1 
 132 


In [4]:
# 4. Módulo 10: Execucao das 3 consultas de trabalho do projeto
vetor_consulta <- function(termos, vocab, idf) {
  q <- as.integer(table(factor(termos, levels = vocab)))
  q * idf
}

ranking_cosseno <- function(consulta, ix) {
  qw <- vetor_consulta(tokenizar(consulta), ix$vocab, ix$idf_tfidf)
  s  <- apply(ix$w, 2, function(d) cosseno(qw, d))
  sort(s, decreasing = TRUE)
}

# Execucao da Consulta 1: 'porto de santos'
r1 <- ranking_cosseno("porto de santos", ix)
cat("=== Consulta 1: 'porto de santos' ===\n")
print(round(r1[1:5], 3))
cat("Top 1 (", names(r1)[1], " - ", origem[[names(r1)[1]]], "): ", substr(docs[[names(r1)[1]]], 1, 140), "...\n\n", sep="")

# Execucao da Consulta 2: 'praias e turismo'
r2 <- ranking_cosseno("praias e turismo", ix)
cat("=== Consulta 2: 'praias e turismo' ===\n")
print(round(r2[1:5], 3))
cat("Top 1 (", names(r2)[1], " - ", origem[[names(r2)[1]]], "): ", substr(docs[[names(r2)[1]]], 1, 140), "...\n\n", sep="")

# Execucao da Consulta 3: 'forte sao joao'
r3 <- ranking_cosseno("forte sao joao", ix)
if (sum(r3 > 0) == 0) r3 <- ranking_cosseno("forte são joão", ix)
cat("=== Consulta 3: 'forte sao joao' ===\n")
print(round(r3[1:5], 3))
cat("Top 1 (", names(r3)[1], " - ", origem[[names(r3)[1]]], "): ", substr(docs[[names(r3)[1]]], 1, 140), "...\n", sep="")

=== Consulta 1: 'porto de santos' ===
  d76   d78   d25   d39   d50 
0.278 0.181 0.138 0.129 0.089 
Top 1 (d76 - Santos): Santos possui o maior porto da América Latina, o Porto de Santos. Suas obras começaram em 1888 e o primeiro trecho de cais foi inaugurado em 1892...

=== Consulta 2: 'praias e turismo' ===
   d2  d118   d20   d51   d18 
0.188 0.164 0.114 0.111 0.105 
Top 1 (d2 - Guaruja): Atualmente, a cidade de Guarujá é conhecida como a "Pérola do Atlântico", devido às suas belas praias e belezas naturais. Muito procurada pelos turistas...

=== Consulta 3: 'forte sao joao' ===
  d117   d121   d132    d16   d123 
 0.296  0.173  0.117  0.108  0.089 
Top 1 (d117 - Bertioga): Umas das cidades mais antigas do Brasil, preserva muitos edifícios históricos, como o Forte de São João, o primeiro forte do país, e destaca-se como...


In [5]:
# 5. Módulo 11: Investigacao de falha do cosseno por sinonimia ortogonal
cat("=== Módulo 11: Investigacao de Falha Ortogonal (sinonimo 'litoral') ===\n")
cat("Termo 'litoral' existe no vocabulario?", "litoral" %in% ix$vocab, "\n")
r_falha <- ranking_cosseno("litoral", ix)
print(round(r_falha[1:5], 3))
cat("Documentos ricos sobre 'praia' recebem score 0 para 'litoral' devido a ortogonalidade dimensional.\n")

=== Módulo 11: Investigacao de Falha Ortogonal (sinonimo 'litoral') ===
Termo 'litoral' existe no vocabulario? TRUE 
  d67    d1   d10   d32   d30 
0.220 0.099 0.096 0.095 0.084 
Documentos ricos sobre 'praia' recebem score 0 para 'litoral' devido a ortogonalidade dimensional.


In [6]:
# 6. Passos de fechamento e anexacao de estado no Colab
if (file.exists("00_FICHA_PROJETO.md")) {
  anexar_estado("00_FICHA_PROJETO.md")
  cat("Estado anexado a 00_FICHA_PROJETO.md\n")
}
if (file.exists("aula02_parteD_consolidado.md")) {
  anexar_estado("aula02_parteD_consolidado.md")
  cat("Estado anexado a aula02_parteD_consolidado.md\n")
}
cat("Sessao finalizada com sucesso!\n")

Estado anexado a 00_FICHA_PROJETO.md
Estado anexado a aula02_parteD_consolidado.md
Sessao finalizada com sucesso!
